# Threshold-Tuned Deep Models and Final Model Selection

LRNG: National (India) | CC: Human Values | SDG: 3 | Data: Time series | Thrust: Intelligent Edge Technologies

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import data_loading.v2_loader as v2

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


Loading PADS 100 Hz dataset, engineered features, fold splits, and metadata

In [2]:
pads_data, pads_meta = v2.load_v1_pads_data(results_dir=os.path.join("..", "results"))
Xd = pads_data
pads_features = np.load(os.path.join("..", "results", "pads_features_100hz.npy"))
splits = v2.load_fold_splits(results_dir=os.path.join("..", "results"))
task_one_hot = v2.get_task_one_hot(pads_meta)

y_pads = pads_meta["cohort_label"].map({"Normal": 0, "Parkinsons-like": 1, "ET-like": 2}).values
groups = pads_meta["subject_id"].values
unique_subs = pads_meta["subject_id"].unique()
sub_true = np.array([y_pads[groups == s][0] for s in unique_subs])
target_names = ["Normal", "Parkinsons-like", "ET-like"]
sub_to_idx = {s: i for i, s in enumerate(unique_subs)}

print("Windows shape:", Xd.shape)
print("Features shape:", pads_features.shape)
print("Total subjects:", len(unique_subs))


Windows shape: (25500, 300, 6)
Features shape: (25500, 30)
Total subjects: 375


Defining neural network architectures: Conv1D+BiLSTM, Combination Model, and Attention-Pooling Head

In [3]:
def build_plain_bilstm(input_shape=(300, 6), num_classes=3):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, kernel_size=5, strides=2, padding="same", activation="relu")(inp)
    x = layers.Conv1D(32, kernel_size=3, strides=2, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(16))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_combination_model(input_shape=(300, 6), feature_dim=30, num_classes=3):
    sig_inp = layers.Input(shape=input_shape)
    x_sig = layers.Conv1D(32, kernel_size=5, strides=2, padding="same", activation="relu")(sig_inp)
    x_sig = layers.Conv1D(32, kernel_size=3, strides=2, padding="same", activation="relu")(x_sig)
    x_sig = layers.MaxPooling1D(2)(x_sig)
    x_sig = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x_sig)
    x_sig = layers.Dropout(0.3)(x_sig)
    x_sig = layers.Bidirectional(layers.LSTM(16))(x_sig)
    x_sig = layers.Dropout(0.3)(x_sig)
    
    feat_inp = layers.Input(shape=(feature_dim,))
    x_feat = layers.Dense(32, activation="relu")(feat_inp)
    x_feat = layers.Dropout(0.3)(x_feat)
    
    merged = layers.Concatenate()([x_sig, x_feat])
    h = layers.Dense(32, activation="relu")(merged)
    out = layers.Dense(num_classes, activation="softmax")(h)
    model = models.Model(inputs=[sig_inp, feat_inp], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_task_aware_window_model(input_shape=(300, 6), task_dim=5, num_classes=3):
    imu_in = layers.Input(shape=input_shape)
    task_in = layers.Input(shape=(task_dim,))
    
    x = layers.Conv1D(32, kernel_size=5, strides=2, padding="same", activation="relu")(imu_in)
    x = layers.Conv1D(32, kernel_size=3, strides=2, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(16))(x)
    x = layers.Dropout(0.3)(x)
    
    t = layers.Dense(16, activation="relu")(task_in)
    concat = layers.Concatenate()([x, t])
    emb = layers.Dense(32, activation="relu")(concat)
    out = layers.Dense(num_classes, activation="softmax")(emb)
    
    model = models.Model(inputs=[imu_in, task_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_attention_pooling_head(num_windows=68, emb_dim=32, num_classes=3):
    inp = layers.Input(shape=(num_windows, emb_dim))
    score = layers.Dense(16, activation="tanh")(inp)
    score = layers.Dense(1)(score)
    weights = layers.Softmax(axis=1)(score)
    pooled = layers.Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1))([inp, weights])
    x = layers.Dense(32, activation="relu")(pooled)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Architectures compiled successfully.")


Architectures compiled successfully.


Executing identically evaluated nested threshold and class-prior tuning across 3 seeds x 5 folds

In [4]:
cache_path = os.path.join("..", "results", "v2_nb11_probs_cache.npz")
split_seeds = [42, 43, 44]
w0_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
w2_grid = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]

def tune_prior_weights(inner_p, inner_true):
    best_f1 = -1.0
    best_w = (1.0, 1.0, 1.0)
    for w0 in w0_grid:
        for w2 in w2_grid:
            scaled = inner_p * np.array([w0, 1.0, w2])
            pred = np.argmax(scaled, axis=1)
            score = f1_score(inner_true, pred, average="macro")
            if score > best_f1:
                best_f1 = score
                best_w = (w0, 1.0, w2)
    return best_w

model_names = [
    "Untuned Random Forest",
    "Tuned Random Forest",
    "Untuned Plain Conv1D+BiLSTM",
    "Tuned Plain Conv1D+BiLSTM",
    "Untuned Combination Model",
    "Tuned Combination Model",
    "Untuned Attention-Pooling",
    "Tuned Attention-Pooling"
]

all_sub_raw_probs = {m: [] for m in model_names}
all_sub_scaled_probs = {m: [] for m in model_names}

if os.path.exists(cache_path):
    print(f"Loading cached evaluations from {cache_path}...")
    cache = np.load(cache_path)
    for m in model_names:
        if f"raw_{m}" in cache and len(cache[f"raw_{m}"]) > 0:
            all_sub_raw_probs[m] = list(cache[f"raw_{m}"])
        if f"scaled_{m}" in cache and len(cache[f"scaled_{m}"]) > 0:
            all_sub_scaled_probs[m] = list(cache[f"scaled_{m}"])
    print("Loaded cached predictions successfully.")
else:
    w0_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
    w2_grid = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
    
    def tune_prior_weights(inner_p, inner_true):
        best_f1 = -1.0
        best_w = (1.0, 1.0, 1.0)
        for w0 in w0_grid:
            for w2 in w2_grid:
                scaled = inner_p * np.array([w0, 1.0, w2])
                pred = np.argmax(scaled, axis=1)
                score = f1_score(inner_true, pred, average="macro")
                if score > best_f1:
                    best_f1 = score
                    best_w = (w0, 1.0, w2)
        return best_w
    
    model_names = [
        "Untuned Random Forest",
        "Tuned Random Forest",
        "Untuned Plain Conv1D+BiLSTM",
        "Tuned Plain Conv1D+BiLSTM",
        "Untuned Combination Model",
        "Tuned Combination Model",
        "Untuned Attention-Pooling",
        "Tuned Attention-Pooling"
    ]
    
    all_sub_raw_probs = {m: [] for m in model_names}
    all_sub_scaled_probs = {m: [] for m in model_names}
    all_win_scaled_probs = {m: [] for m in model_names if "Attention" not in m}
    
    for s in split_seeds:
        print(f"\n--- Running Split Seed {s} (5 Folds) ---")
        tf.random.set_seed(s)
        np.random.seed(s)
        
        oof_rf_raw = np.zeros((len(y_pads), 3), dtype=np.float32)
        oof_rf_scaled = np.zeros((len(y_pads), 3), dtype=np.float32)
        
        oof_bilstm_raw = np.zeros((len(y_pads), 3), dtype=np.float32)
        oof_bilstm_scaled = np.zeros((len(y_pads), 3), dtype=np.float32)
        
        oof_combo_raw = np.zeros((len(y_pads), 3), dtype=np.float32)
        oof_combo_scaled = np.zeros((len(y_pads), 3), dtype=np.float32)
        
        oof_attn_raw = np.zeros((len(unique_subs), 3), dtype=np.float32)
        oof_attn_scaled = np.zeros((len(unique_subs), 3), dtype=np.float32)
        
        for fold in range(5):
            tr_idx = splits[f"split_{s}_tr_{fold}"]
            te_idx = splits[f"split_{s}_te_{fold}"]
            
            ch_std = Xd[tr_idx].std(axis=(0, 1))
            ch_std[ch_std == 0] = 1.0
            Xn_tr = (Xd[tr_idx] / ch_std).astype(np.float32)
            Xn_te = (Xd[te_idx] / ch_std).astype(np.float32)
            
            tr_subs = np.unique(groups[tr_idx])
            te_subs = np.unique(groups[te_idx])
            y_sub_tr = np.array([y_pads[groups == sub][0] for sub in tr_subs])
            
            inner_sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=s)
            in_tr_local, in_va_local = next(inner_sgkf.split(pads_features[tr_idx], y_pads[tr_idx], groups[tr_idx]))
            in_tr = tr_idx[in_tr_local]
            in_va = tr_idx[in_va_local]
            in_va_subs = np.unique(groups[in_va])
            in_va_sub_true = np.array([y_pads[groups == sub][0] for sub in in_va_subs])
            
            # 1. Random Forest
            rf_in = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, random_state=s, n_jobs=-1, class_weight="balanced")
            rf_in.fit(pads_features[in_tr], y_pads[in_tr])
            rf_in_p = rf_in.predict_proba(pads_features[in_va])
            rf_in_sub_p = np.array([np.mean(rf_in_p[groups[in_va] == sub], axis=0) for sub in in_va_subs])
            w_rf = tune_prior_weights(rf_in_sub_p, in_va_sub_true)
            
            rf_full = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, random_state=s, n_jobs=-1, class_weight="balanced")
            rf_full.fit(pads_features[tr_idx], y_pads[tr_idx])
            rf_te_p = rf_full.predict_proba(pads_features[te_idx])
            oof_rf_raw[te_idx] = rf_te_p
            oof_rf_scaled[te_idx] = rf_te_p * np.array(w_rf)
            
            # 2. Plain Conv1D + BiLSTM
            cw = compute_class_weight("balanced", classes=np.unique(y_pads[tr_idx]), y=y_pads[tr_idx])
            cw_dict = dict(enumerate(cw))
            
            tf.keras.backend.clear_session()
            m_bilstm_in = build_plain_bilstm()
            m_bilstm_in.fit(Xn_tr[in_tr_local], y_pads[in_tr], epochs=6, batch_size=512, class_weight=cw_dict, verbose=0)
            bilstm_in_p = m_bilstm_in.predict(Xn_tr[in_va_local], batch_size=1024, verbose=0)
            bilstm_in_sub_p = np.array([np.mean(bilstm_in_p[groups[in_va] == sub], axis=0) for sub in in_va_subs])
            w_bilstm = tune_prior_weights(bilstm_in_sub_p, in_va_sub_true)
            
            tf.keras.backend.clear_session()
            m_bilstm = build_plain_bilstm()
            m_bilstm.fit(Xn_tr, y_pads[tr_idx], epochs=7, batch_size=512, class_weight=cw_dict, verbose=0)
            bilstm_te_p = m_bilstm.predict(Xn_te, batch_size=1024, verbose=0)
            oof_bilstm_raw[te_idx] = bilstm_te_p
            oof_bilstm_scaled[te_idx] = bilstm_te_p * np.array(w_bilstm)
            
            # 3. Combination Model
            feat_tr = pads_features[tr_idx]
            feat_te = pads_features[te_idx]
            feat_std = feat_tr.std(axis=0)
            feat_std[feat_std == 0] = 1.0
            feat_tr_n = (feat_tr - feat_tr.mean(axis=0)) / feat_std
            feat_te_n = (feat_te - feat_tr.mean(axis=0)) / feat_std
            
            tf.keras.backend.clear_session()
            m_combo_in = build_combination_model()
            m_combo_in.fit([Xn_tr[in_tr_local], feat_tr_n[in_tr_local]], y_pads[in_tr], epochs=6, batch_size=512, class_weight=cw_dict, verbose=0)
            combo_in_p = m_combo_in.predict([Xn_tr[in_va_local], feat_tr_n[in_va_local]], batch_size=1024, verbose=0)
            combo_in_sub_p = np.array([np.mean(combo_in_p[groups[in_va] == sub], axis=0) for sub in in_va_subs])
            w_combo = tune_prior_weights(combo_in_sub_p, in_va_sub_true)
            
            tf.keras.backend.clear_session()
            m_combo = build_combination_model()
            m_combo.fit([Xn_tr, feat_tr_n], y_pads[tr_idx], epochs=7, batch_size=512, class_weight=cw_dict, verbose=0)
            combo_te_p = m_combo.predict([Xn_te, feat_te_n], batch_size=1024, verbose=0)
            oof_combo_raw[te_idx] = combo_te_p
            oof_combo_scaled[te_idx] = combo_te_p * np.array(w_combo)
            
            # 4. Attention-Pooling Model
            task_tr = task_one_hot[tr_idx]
            task_te = task_one_hot[te_idx]
            
            tf.keras.backend.clear_session()
            m_win = build_task_aware_window_model()
            m_win.fit([Xn_tr, task_tr], y_pads[tr_idx], epochs=7, batch_size=512, class_weight=cw_dict, verbose=0)
            encoder = models.Model(inputs=m_win.inputs, outputs=m_win.layers[-2].output)
            embs_tr = encoder.predict([Xn_tr, task_tr], batch_size=1024, verbose=0)
            embs_te = encoder.predict([Xn_te, task_te], batch_size=1024, verbose=0)
            
            H_tr = np.array([embs_tr[groups[tr_idx] == sub] for sub in tr_subs])
            H_te = np.array([embs_te[groups[te_idx] == sub] for sub in te_subs])
            
            sub_cw = compute_class_weight("balanced", classes=np.unique(y_sub_tr), y=y_sub_tr)
            sub_cw_dict = dict(enumerate(sub_cw))
            
            inner_sub_sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=s)
            in_sub_tr, in_sub_va = next(inner_sub_sgkf.split(H_tr, y_sub_tr, tr_subs))
            
            m_attn_in = build_attention_pooling_head()
            m_attn_in.fit(H_tr[in_sub_tr], y_sub_tr[in_sub_tr], epochs=10, batch_size=64, class_weight=sub_cw_dict, verbose=0)
            attn_in_p = m_attn_in.predict(H_tr[in_sub_va], verbose=0)
            w_attn = tune_prior_weights(attn_in_p, y_sub_tr[in_sub_va])
            
            m_attn = build_attention_pooling_head()
            m_attn.fit(H_tr, y_sub_tr, epochs=12, batch_size=64, class_weight=sub_cw_dict, verbose=0)
            attn_te_p = m_attn.predict(H_te, verbose=0)
            
            for sub, p in zip(te_subs, attn_te_p):
                idx = sub_to_idx[sub]
                oof_attn_raw[idx] = p
                oof_attn_scaled[idx] = p * np.array(w_attn)
                
            print(f"Fold {fold} finished. Tuned weights: RF={w_rf}, BiLSTM={w_bilstm}, Combo={w_combo}, Attn={w_attn}")
            
        for sub in unique_subs:
            pass
            
        sub_rf_raw = np.array([np.mean(oof_rf_raw[groups == sub], axis=0) for sub in unique_subs])
        sub_rf_scaled = np.array([np.mean(oof_rf_scaled[groups == sub], axis=0) for sub in unique_subs])
        
        sub_bilstm_raw = np.array([np.mean(oof_bilstm_raw[groups == sub], axis=0) for sub in unique_subs])
        sub_bilstm_scaled = np.array([np.mean(oof_bilstm_scaled[groups == sub], axis=0) for sub in unique_subs])
        
        sub_combo_raw = np.array([np.mean(oof_combo_raw[groups == sub], axis=0) for sub in unique_subs])
        sub_combo_scaled = np.array([np.mean(oof_combo_scaled[groups == sub], axis=0) for sub in unique_subs])
        
        all_sub_raw_probs["Untuned Random Forest"].append(sub_rf_raw)
        all_sub_scaled_probs["Tuned Random Forest"].append(sub_rf_scaled)
        all_win_scaled_probs["Tuned Random Forest"].append(oof_rf_scaled)
        
        all_sub_raw_probs["Untuned Plain Conv1D+BiLSTM"].append(sub_bilstm_raw)
        all_sub_scaled_probs["Tuned Plain Conv1D+BiLSTM"].append(sub_bilstm_scaled)
        all_win_scaled_probs["Tuned Plain Conv1D+BiLSTM"].append(oof_bilstm_scaled)
        
        all_sub_raw_probs["Untuned Combination Model"].append(sub_combo_raw)
        all_sub_scaled_probs["Tuned Combination Model"].append(sub_combo_scaled)
        all_win_scaled_probs["Tuned Combination Model"].append(oof_combo_scaled)
        
        all_sub_raw_probs["Untuned Attention-Pooling"].append(oof_attn_raw)
        all_sub_scaled_probs["Tuned Attention-Pooling"].append(oof_attn_scaled)
        
        print(f"Seed {s} evaluation complete.")
    
cache_path = os.path.join("..", "results", "v2_nb11_probs_cache.npz")
save_dict = {}
for m in model_names:
    save_dict[f"raw_{m}"] = np.array(all_sub_raw_probs[m])
    save_dict[f"scaled_{m}"] = np.array(all_sub_scaled_probs[m])
np.savez_compressed(cache_path, **save_dict)
print(f"Saved evaluation cache to {cache_path}")


Loading cached evaluations from ..\results\v2_nb11_probs_cache.npz...
Loaded cached predictions successfully.
Saved evaluation cache to ..\results\v2_nb11_probs_cache.npz


Computing subject-bootstrap confidence intervals, per-class recalls, and one-vs-rest AUROCs

In [5]:
results_table = []
detailed_recalls = {}

eval_pairs = [
    ("Untuned Random Forest", all_sub_raw_probs["Untuned Random Forest"], False),
    ("Tuned Random Forest", all_sub_scaled_probs["Tuned Random Forest"], True),
    ("Untuned Plain Conv1D+BiLSTM", all_sub_raw_probs["Untuned Plain Conv1D+BiLSTM"], False),
    ("Tuned Plain Conv1D+BiLSTM", all_sub_scaled_probs["Tuned Plain Conv1D+BiLSTM"], True),
    ("Untuned Combination Model", all_sub_raw_probs["Untuned Combination Model"], False),
    ("Tuned Combination Model", all_sub_scaled_probs["Tuned Combination Model"], True),
    ("Untuned Attention-Pooling", all_sub_raw_probs["Untuned Attention-Pooling"], False),
    ("Tuned Attention-Pooling", all_sub_scaled_probs["Tuned Attention-Pooling"], True)
]

for name, seed_probs, is_tuned in eval_pairs:
    avg_sub_p = np.mean(seed_probs, axis=0)
    sub_pred = np.argmax(avg_sub_p, axis=1)
    
    macro_f1 = float(f1_score(sub_true, sub_pred, average="macro"))
    f1_low, f1_high = v2.compute_bootstrap_ci(sub_true, avg_sub_p, lambda y, p: f1_score(y, np.argmax(p, axis=1), average="macro"), n_boot=1000, seed=42)
    
    bacc = float(balanced_accuracy_score(sub_true, sub_pred))
    bacc_low, bacc_high = v2.compute_bootstrap_ci(sub_true, avg_sub_p, lambda y, p: balanced_accuracy_score(y, np.argmax(p, axis=1)), n_boot=1000, seed=42)
    
    cm = confusion_matrix(sub_true, sub_pred, labels=[0, 1, 2])
    rec_norm = cm[0, 0] / cm[0].sum()
    rec_pd = cm[1, 1] / cm[1].sum()
    rec_et = cm[2, 2] / cm[2].sum()
    detailed_recalls[name] = (rec_norm, rec_pd, rec_et, cm)
    
    y_sub_oh = np.eye(3)[sub_true]
    sub_p_norm = avg_sub_p / np.sum(avg_sub_p, axis=1, keepdims=True)
    auc_norm = float(roc_auc_score(y_sub_oh[:, 0], sub_p_norm[:, 0]))
    auc_pd = float(roc_auc_score(y_sub_oh[:, 1], sub_p_norm[:, 1]))
    auc_et = float(roc_auc_score(y_sub_oh[:, 2], sub_p_norm[:, 2]))
    auc_macro = float(roc_auc_score(y_sub_oh, sub_p_norm, average="macro", multi_class="ovr"))
    
    results_table.append({
        "Model": name,
        "Subject_Macro_F1": macro_f1,
        "Subject_F1_CI_Low": f1_low,
        "Subject_F1_CI_High": f1_high,
        "Subject_Balanced_Acc": bacc,
        "Subject_BAcc_CI_Low": bacc_low,
        "Subject_BAcc_CI_High": bacc_high,
        "AUROC_Normal": auc_norm,
        "AUROC_PD": auc_pd,
        "AUROC_ET": auc_et,
        "AUROC_Macro": auc_macro,
        "Recall_Normal": rec_norm,
        "Recall_PD": rec_pd,
        "Recall_ET": rec_et
    })

df_final = pd.DataFrame(results_table)
os.makedirs(os.path.join("..", "results", "metrics"), exist_ok=True)
df_final.to_csv(os.path.join("..", "results", "metrics", "v2_track1_all_models_threshold_tuned.csv"), index=False)

print("=== FINAL COMPARISON TABLE (Protocol: Same 3 seeds x 5 folds, inner fold threshold-tuning) ===")
print(df_final[["Model", "Subject_Macro_F1", "Subject_F1_CI_Low", "Subject_F1_CI_High", "Subject_Balanced_Acc", "AUROC_Macro", "Recall_Normal", "Recall_PD", "Recall_ET"]].to_string(index=False))


=== FINAL COMPARISON TABLE (Protocol: Same 3 seeds x 5 folds, inner fold threshold-tuning) ===
                      Model  Subject_Macro_F1  Subject_F1_CI_Low  Subject_F1_CI_High  Subject_Balanced_Acc  AUROC_Macro  Recall_Normal  Recall_PD  Recall_ET
      Untuned Random Forest          0.622402           0.556996            0.683075              0.668320     0.837528       0.784810   0.720149   0.500000
        Tuned Random Forest          0.593018           0.523429            0.654009              0.669741     0.804427       0.784810   0.652985   0.571429
Untuned Plain Conv1D+BiLSTM          0.460422           0.404632            0.515842              0.629050     0.750439       0.835443   0.373134   0.678571
  Tuned Plain Conv1D+BiLSTM          0.387953           0.335207            0.438775              0.597422     0.711503       0.835443   0.242537   0.714286
  Untuned Combination Model          0.487668           0.426944            0.541176              0.658100     0.785742 

Documenting the DAE pretraining experiment as a negative finding with reconstruction MSE baseline comparison

In [6]:
dae_recon_mse = 0.57768
zero_baseline_mse = 1.00895
mse_reduction = (zero_baseline_mse - dae_recon_mse) / zero_baseline_mse

print("=== DAE Pretraining Negative Finding Summary ===")
print(f"Predict-Zeros Baseline MSE: {zero_baseline_mse:.5f}")
print(f"DAE Reconstruction MSE:     {dae_recon_mse:.5f}")
print(f"Relative MSE Reduction:     {mse_reduction * 100:.2f}%")
print("Downstream Subject Macro-F1: Scratch = 0.5064, Frozen DAE = 0.3740, Fine-Tuned DAE = 0.4865")
print("Outcome: Pretraining provided no representation gain over supervised training from scratch.")


=== DAE Pretraining Negative Finding Summary ===
Predict-Zeros Baseline MSE: 1.00895
DAE Reconstruction MSE:     0.57768
Relative MSE Reduction:     42.74%
Downstream Subject Macro-F1: Scratch = 0.5064, Frozen DAE = 0.3740, Fine-Tuned DAE = 0.4865
Outcome: Pretraining provided no representation gain over supervised training from scratch.


Applying pre-registered selection rule, selecting submitted deep model, and exporting named TFLite models

In [7]:
best_deep_model_name = "Strided_Conv1D_BiLSTM"
print("Pre-registered Selection Policy:")
print("1. Models with overlapping 95% bootstrap confidence intervals are considered statistically tied.")
print("2. When performance intervals overlap, the simpler architecture with minimal operational assumptions is selected.")
print("3. A deep model must be submitted for edge deployment (excluding tabular Random Forest).")
print(f"Selected Deep Model: {best_deep_model_name}")

if "Attention" in best_deep_model_name:
    print("Guided test mapping: 3 tasks, 1 wrist maps to 12 windows, requiring zero-padding or dynamic batching across 68 slots.")
    print("Distribution shift: Model was trained on all 5 tasks and both wrists (68 windows). Guided test presents domain shift.")
else:
    print("Direct Streaming: The Strided Conv1D + BiLSTM operates window-by-window (300 samples = 3 s), independent of task sequence or wrist count.")

tf.keras.backend.clear_session()
def build_exportable_model(input_shape=(300, 6), num_classes=3):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, kernel_size=5, strides=2, padding="same", activation="relu")(inp)
    x = layers.Conv1D(32, kernel_size=3, strides=2, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True, unroll=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(16, unroll=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

final_model = build_exportable_model()
cw = compute_class_weight("balanced", classes=np.unique(y_pads), y=y_pads)
cw_dict = dict(enumerate(cw))

ch_std_full = Xd.std(axis=(0, 1))
ch_std_full[ch_std_full == 0] = 1.0
Xn_full = (Xd / ch_std_full).astype(np.float32)

final_model.fit(Xn_full, y_pads, epochs=10, batch_size=512, class_weight=cw_dict, verbose=0)

keras_path = os.path.join("..", "results", "models", f"{best_deep_model_name}.keras")
final_model.save(keras_path)

converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

tflite_path = os.path.join("..", "results", "models", f"{best_deep_model_name}.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

converter_quant = tf.lite.TFLiteConverter.from_keras_model(final_model)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_quant.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_quant_model = converter_quant.convert()

tflite_quant_path = os.path.join("..", "results", "models", f"{best_deep_model_name}_quant.tflite")
with open(tflite_quant_path, "wb") as f:
    f.write(tflite_quant_model)

print(f"Saved {keras_path} ({os.path.getsize(keras_path)/1024:.2f} KB)")
print(f"Saved {tflite_path} ({os.path.getsize(tflite_path)/1024:.2f} KB, pure TFLite)")
print(f"Saved {tflite_quant_path} ({os.path.getsize(tflite_quant_path)/1024:.2f} KB, quantized)")


Pre-registered Selection Policy:
1. Models with overlapping 95% bootstrap confidence intervals are considered statistically tied.
2. When performance intervals overlap, the simpler architecture with minimal operational assumptions is selected.
3. A deep model must be submitted for edge deployment (excluding tabular Random Forest).
Selected Deep Model: Strided_Conv1D_BiLSTM
Direct Streaming: The Strided Conv1D + BiLSTM operates window-by-window (300 samples = 3 s), independent of task sequence or wrist count.


INFO:tensorflow:Assets written to: C:\Users\ESHWAR~1\AppData\Local\Temp\tmpy0f4n9x1\assets


INFO:tensorflow:Assets written to: C:\Users\ESHWAR~1\AppData\Local\Temp\tmpy0f4n9x1\assets


INFO:tensorflow:Assets written to: C:\Users\ESHWAR~1\AppData\Local\Temp\tmpe81r3zlt\assets


INFO:tensorflow:Assets written to: C:\Users\ESHWAR~1\AppData\Local\Temp\tmpe81r3zlt\assets


Saved ..\results\models\Strided_Conv1D_BiLSTM.keras (456.51 KB)
Saved ..\results\models\Strided_Conv1D_BiLSTM.tflite (556.58 KB, pure TFLite)
Saved ..\results\models\Strided_Conv1D_BiLSTM_quant.tflite (468.48 KB, quantized)
